# Scikit-Learn Integration with Venn-ABERS Calibrator

This notebook demonstrates how the `venn-abers-scikit` library integrates natively with the `scikit-learn` ecosystem, focusing on **scikit-learn Pipelines**.
We will compare the Inductive (IVAP) and Cross (CVAP) Venn-ABERS calibrators against standard scikit-learn calibration methods (Platt scaling / sigmoid, and Isotonic Regression) entirely within Pipelines.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss

from venn_abers import VennAbersCalibrator, VennAberRegressor

import warnings
warnings.filterwarnings("ignore")

## 1. Classification & Calibration inside Pipelines

First, let's create a synthetic dataset for binary classification that requires scaling.

In [2]:
X, y = make_classification(n_samples=2000, n_classes=2, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


### Standard Scikit-Learn Calibration (Sigmoid & Isotonic)
We will fit standard Scikit-Learn `CalibratedClassifierCV` models using both `sigmoid` and `isotonic` methods as the final step of a pipeline.

In [3]:
clf = GaussianNB()

# Sigmoid Calibration Pipeline
pipeline_sigmoid = Pipeline([
    ('scaler', StandardScaler()),
    ('calibrated_clf', CalibratedClassifierCV(clf, method='sigmoid', cv=5))
])
pipeline_sigmoid.fit(X_train, y_train)
p_pred_sigmoid = pipeline_sigmoid.predict_proba(X_test)

# Isotonic Calibration Pipeline
pipeline_isotonic = Pipeline([
    ('scaler', StandardScaler()),
    ('calibrated_clf', CalibratedClassifierCV(clf, method='isotonic', cv=5))
])
pipeline_isotonic.fit(X_train, y_train)
p_pred_isotonic = pipeline_isotonic.predict_proba(X_test)


### Venn-ABERS Calibration (Inductive and Cross-Validated)
Now we'll use `vA` natively within `sklearn.pipeline.Pipeline`, showing that it correctly scales and processes.

In [4]:
# Inductive Venn-ABERS (IVAP) Pipeline
pipeline_ivap = Pipeline([
    ('scaler', StandardScaler()),
    ('va', VennAbersCalibrator(estimator=GaussianNB(), inductive=True, cal_size=0.2, random_state=42))
])
pipeline_ivap.fit(X_train, y_train)
p_pred_ivap = pipeline_ivap.predict_proba(X_test)

# Cross Venn-ABERS (CVAP) Pipeline
pipeline_cvap = Pipeline([
    ('scaler', StandardScaler()),
    ('va', VennAbersCalibrator(estimator=GaussianNB(), inductive=False, n_splits=5, random_state=42))
])
pipeline_cvap.fit(X_train, y_train)
p_pred_cvap = pipeline_cvap.predict_proba(X_test)


### Evaluation Metrics
Let's compare the pipeline methods using Brier score and Log Loss.

In [5]:
def print_metrics(name, y_true, p_pred):
    prob_pos = p_pred[:, 1]
    brier = brier_score_loss(y_true, prob_pos)
    logloss = log_loss(y_true, p_pred)
    print(f"{name:<20} | Brier Score: {brier:.4f} | Log Loss: {logloss:.4f}")

print_metrics("Pipeline Sigmoid", y_test, p_pred_sigmoid)
print_metrics("Pipeline Isotonic", y_test, p_pred_isotonic)
print_metrics("Pipeline IVAP", y_test, p_pred_ivap)
print_metrics("Pipeline CVAP", y_test, p_pred_cvap)


Pipeline Sigmoid     | Brier Score: 0.1310 | Log Loss: 0.4147
Pipeline Isotonic    | Brier Score: 0.1312 | Log Loss: 0.4121
Pipeline IVAP        | Brier Score: 0.1314 | Log Loss: 0.4083
Pipeline CVAP        | Brier Score: 0.1305 | Log Loss: 0.4102


## 2. Venn-ABERS Regressor Pipeline Example

Here is an example using `VennAberRegressor` inside an `sklearn.pipeline.Pipeline` on a synthetic regression dataset.

In [6]:
X_reg, y_reg = make_regression(n_samples=500, n_features=10, noise=0.1, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

regressor = DecisionTreeRegressor(max_depth=5, random_state=42)

# Inductive Venn-ABERS Regressor inside a Pipeline
pipeline_var = Pipeline([
    ('scaler', StandardScaler()),
    ('var', VennAberRegressor(estimator=regressor, inductive=True, cal_size=0.2, random_state=42))
])
pipeline_var.fit(X_train_r, y_train_r)

# We can call predict safely through the pipeline wrapper.
# (Note: scikit-learn standard pipelines do not naturally unpack tuple return types yet but the step itself does)
y_pred, y_bounds = pipeline_var.predict(X_test_r)

print(f"Predictions generated: {len(y_pred)}")
print(f"First 3 estimates:\n{y_pred[:3]}\n")
print(f"First 3 bounds:\n{y_bounds[:3]}")


Predictions generated: 100
First 3 estimates:
[-12.64361391 -18.22463911  73.63282545]

First 3 bounds:
[[-30.10102584   0.77740411]
 [-30.10102584  -9.35759801]
 [ 49.59993535 108.89486463]]
